# Individual-Differences Data Analysis

Standalone quality audit and descriptive analysis of the MECO-L2 participant measures. This notebook intentionally does not merge comprehension, fixation, or reading-rate data.

## Scope

The goal is to determine whether the individual-difference data require preprocessing, identify redundant measures, and assess whether standardized composites are defensible for a future integrated analysis.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from analysis.individual_differences.io import MEASURES, load_data, missingness_summary, language_summary
from analysis.individual_differences.features import add_composites, correlation_table, standardize_within_wave
from analysis.individual_differences.plots import plot_correlations, plot_distributions, plot_language_profiles, plot_missingness

OUTPUT = ROOT / 'analysis' / 'individual_differences' / 'output' / 'notebook'
PLOTS = OUTPUT / 'plots'
OUTPUT.mkdir(parents=True, exist_ok=True)
PLOTS.mkdir(parents=True, exist_ok=True)

## Dataset Overview

In [ ]:
data = load_data('both')
print(f'Rows: {len(data):,}')
print(f'Participants: {data.participant_id.nunique():,}')
print(f'Measures: {len(MEASURES)}')
display(data.groupby(['wave', 'lang'], observed=True).size().reset_index(name='participants'))
display(data[MEASURES].describe().T)

## Missingness and Quality

In [ ]:
display(missingness_summary(data))
display(data.groupby('wave', observed=True)['all_measures_complete'].agg(['size', 'sum']).rename(columns={'sum': 'complete_cases'}))
for fig in [plot_missingness(data, PLOTS)]:
    display(fig)
    plt.close(fig)

In [ ]:
quality_columns = ['duplicate_rows', 'id_valid', 'lang_id_consistent', 'numeric_conversion_issue']
available_quality = [column for column in quality_columns if column in data.columns]
display(data[available_quality].agg(['sum', 'mean']).T)
display(data[[column for column in data.columns if column.endswith('_out_of_plausible_range')]].sum())

## Score Distributions

In [ ]:
fig = plot_distributions(data, PLOTS)
display(fig)
plt.close(fig)
fig = plot_language_profiles(data, PLOTS)
display(fig)
plt.close(fig)

## Measure Redundancy

In [ ]:
display(correlation_table(data))
for fig in plot_correlations(data, PLOTS):
    display(fig)
    plt.close(fig)

## Standardized Scores and Composites

Scores are standardized within wave. The language-skill composite is the mean of LexTALE, capped vocabulary, and the two TOWRE measures. Raw scores are not summed because their scales differ.

In [ ]:
standardized = add_composites(standardize_within_wave(data))
display(standardized.groupby('wave', observed=True)[['language_skill_composite', 'cognitive_skill', 'motivation_standardized', 'spelling_standardized']].agg(['count', 'mean', 'std']))
standardized.to_csv(OUTPUT / 'individual_differences_standardized.csv', index=False)

## Conclusion

Use this audit to decide which measures are sufficiently complete and non-redundant for a future integrated analysis. Do not impute or remove flagged values without checking the original test definitions.